# 02 — Состав данных

Описание данных для работы: сколько записей, как разложены по сплитам и классам, сколько записей на атаку, как распределены длительности относительно `t_fixed`.

Ноутбук **тонкий** (§7): логика — в `src/viz/dataset.py`. Читаются манифесты Стадии 0 (§6.1); длительности берутся из числа кадров кэша Стадии 1, потому что колонку `duration` адаптеры не заполняют, а сравнивать с `t_fixed` надо именно кадры.

**Предусловие:** прогнаны Стадия 0 для всех датасетов и Стадия 1 хотя бы частично.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import Paths
from src.data.features import get_extractor
from src.viz import dataset as D
from src.viz.style import apply_style, save_figure

apply_style()
PATHS = Paths(root=ROOT)

DATASETS = ["asvspoof2019_la", "asvspoof2021_la", "asvspoof2021_df", "for-2seconds"]
EXTRACTOR = get_extractor("lfcc")
T_FIXED = 400
SAMPLE = 3000   # сколько файлов кэша просматривать на датасет для гистограммы

manifest = D.load_manifests(PATHS, DATASETS)
print(f"всего записей: {len(manifest)}")

## Сводная таблица

Столбец «спуф, %» — та несбалансированность, из-за которой на Стадии 2 включён `pos_weight`.

In [ ]:
table = D.composition_table(manifest)
table

In [ ]:
fig = D.plot_class_balance(manifest)
save_figure(fig, "data_class_balance", paths=PATHS)

In [ ]:
fig = D.plot_split_composition(manifest)
save_figure(fig, "data_split_composition", paths=PATHS)

## Записей на атаку

Читается вместе с разбивкой EER по атакам из ноутбука 04: EER атаки, у которой десятки записей, шумный, и сравнивать его с атакой на тысячах записей нельзя.

In [ ]:
for ds in DATASETS:
    try:
        fig = D.plot_attack_counts(manifest, ds)
    except ValueError as e:
        print(f"{ds}: {e}")
        continue
    save_figure(fig, f"data_attacks_{ds}", paths=PATHS)

## Длительности и `t_fixed`

Ключевая картинка для интерпретации кросс-датасетных метрик (§8). Линия `t_fixed` делит ось на две области с разным обращением с записью (§7): правее запись обрезается, левее — повторяется до нужной длины.

Если распределение датасета целиком лежит по одну сторону линии, это систематический артефакт домена, а не особенность отдельных файлов, и в работе его надо оговорить.

In [ ]:
counts = {}
for ds in DATASETS:
    try:
        counts[ds] = D.frame_counts(PATHS, EXTRACTOR, ds, sample=SAMPLE)
    except FileNotFoundError as e:
        print(f"{ds}: пропущен — {e}")

fig = D.plot_durations(counts, EXTRACTOR, t_fixed=T_FIXED)
save_figure(fig, "data_durations", paths=PATHS)

In [ ]:
import numpy as np
import pandas as pd

hop, sr = EXTRACTOR.hop_length, EXTRACTOR.target_sr
rows = []
for ds, c in counts.items():
    sec = np.asarray(c) * hop / sr
    rows.append({
        "dataset_id": ds,
        "n": len(c),
        "медиана, с": round(float(np.median(sec)), 2),
        "p05, с": round(float(np.quantile(sec, 0.05)), 2),
        "p95, с": round(float(np.quantile(sec, 0.95)), 2),
        "короче t_fixed, %": round(100 * float((np.asarray(c) < T_FIXED).mean()), 1),
    })
pd.DataFrame(rows)